<a href="https://colab.research.google.com/github/SusanaBN30/Tareas/blob/main/Data_Analyst_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data Analyst Assignment

## Introduction

You are working with a US retail customer on a pilot deployment.  They are using technology to track their merchandise throughout their supply chain.  The flow of their supply is:

*   **DC 1:**  Orders are filled and palletized.
*   **Truck:** Pallets travel from the DC 1 to DC 2 via semi-truck.
*   **DC 2:**  Pallets are unloaded, and additional merchandise may be added.  They are then reloaded onto a new truck.
*   **Truck:** Pallets travel from DC 2 to the Store.
*   **Store:** Pallets are unloaded, cases are removed, and stocked, and the empty cases are left behind the building awaiting pickup.

Your job is to dig into the data and find compelling insights to show the value fo the technology and help move the contract from a pilot into a full scaled deployment.



---

## Part 0: Imports

Import necessary packages and

In [2]:
# YOUR CODE HERE:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
df = pd.read_excel('Assignment_1.xlsx')

### Dataset Overview

* Site:  A large space that could contain multiple readers. Ex: DC 1.
* Zone:  Point of interest. These represent areas in which repeaters are installed. These can be thought of as sub-zones.  Ex: Dock Doors.
* Asset ID: The unique ID of the asset.
* Asset Type: The type of thing that is detected (ie tote).
* Device ID: The unique gateway reader ID that detected the device in the zone (there can be multiple in one zone).
* Time est: The time in EST.
* Lon: Longituge
* Lat: Latitude
* Temperature_C / F: Temperature in Celsius, Fahrentheit

## PART 1: Data Overview

### Question 1:

* How many unique cases were we tracking throughout this pilot. (1 pt)
* What are the unique zones we could see (1 pt)
* How many POI's are in each Zone. (2 pts)


In [13]:
# YOUR CODE HERE:
# 1.
casos_unicos = df['asset_id'].nunique()
print(f"How many unique cases were we tracking throughout this pilot: {casos_unicos}")

# 2.
zonas_unicas = df['Zone'].unique()
print(f"\nWhat are the unique zones we could see: {zonas_unicas}")

# 3.
poi_por_zona = df.groupby('Zone')['device_id'].nunique().reset_index()
poi_por_zona.columns = ['Zone', 'POI Count']
print("\nHow many POI's are in each Zone.")
print(poi_por_zona)

How many unique cases were we tracking throughout this pilot: 18

What are the unique zones we could see: ['dock_doors_DC1' 'dock_doors_DC2' 'Forklift3_DC1' 'pallet_assembly_DC1'
 'PhoneKit1Bridge' 'PhoneKit2Bridge' 'PhoneKit2GW' 'point_of_sale_Store'
 'receiving_Store' 'staging_DC1' 'staging_DC2' 'store_back_Store'
 'store_front_Store' 'Forklift1_DC1' 'PhoneKit1GW']

How many POI's are in each Zone.
                   Zone  POI Count
0         Forklift1_DC1          1
1         Forklift3_DC1          1
2       PhoneKit1Bridge          1
3           PhoneKit1GW          1
4       PhoneKit2Bridge          1
5           PhoneKit2GW          1
6        dock_doors_DC1          4
7        dock_doors_DC2          3
8   pallet_assembly_DC1          2
9   point_of_sale_Store          2
10      receiving_Store          3
11          staging_DC1          5
12          staging_DC2          4
13     store_back_Store          3
14    store_front_Store          6


### Question 2:

* What is the temperature range we see?  (1pt)
* Where is temperature the highest and lowest (1pt)

In [20]:
# YOUR CODE HERE:
# 1.
print(f"Temp Mínima: {df['Temperature_C'].min()}°C")
print(f"Temp Máxima: {df['Temperature_C'].max()}°C")
print(f"What is the temperature range we see? [{df['Temperature_C'].min()}°C,{df['Temperature_C'].max()}°C]")

# 2.
max_temperatura = df.loc[df['Temperature_C'].idxmax()]
min_temperatura = df.loc[df['Temperature_C'].idxmin()]
print(f"\nWhere is temperature the highest and lowest")
print(f"The highest temperature is in {max_temperatura['Site']} ({max_temperatura['Zone']})")
print(f"The lowest temperature is in {min_temperatura['Site']} ({min_temperatura['Zone']})")

Temp Mínima: 19.0°C
Temp Máxima: 44.0°C
What is the temperature range we see? [19.0°C,44.0°C]

Where is temperature the highest and lowest
The highest temperature is in Store (receiving_Store)
The lowest temperature is in Store (store_back_Store)


## Part 2: The Journey of a Case

### Question 3:

* Create a visualization that shows where a case was at over time at the zone or POI level. Imagine that this would be included in your presentation to the customer. (Non techical audience) (3 pts)

In [25]:
# YOUR CODE HERE:
zona_interes = 'pallet_assembly_DC1'
df_filtrado = df_ejemplo[df_ejemplo['Zone'] == zona_interes]

fig3_solo_zona = px.scatter(df_filtrado,
                            x='time_est',
                            y='device_id',
                            color='device_id',
                            title=f'Detalle de Actividad en Zona: {zona_interes}',
                            labels={'time_est': 'Tiempo', 'device_id': 'ID del Sensor'},
                            template='plotly_white')

fig3_solo_zona.update_traces(marker=dict(size=10))
fig3_solo_zona.show()

### Question 4:

* Visualize how the temperatue changes over time along its journey.  Imagine that this would be included in your presentation to the customer. (Non techical audience) (4 pts)



In [26]:
# YOUR CODE HERE:
fig4_solo_zona = px.line(df_filtrado,
                         x='time_est',
                         y='Temperature_C',
                         title=f'Monitoreo de Temperatura en: {zona_interes}',
                         labels={'time_est': 'Tiempo', 'Temperature_C': 'Temp (°C)'},
                         markers=True,
                         line_shape='linear')

fig4_solo_zona.update_traces(line_color='#EF553B')
fig4_solo_zona.show()

### Question 5:
* Visualize the lon lat data on a map to show how the case traveled.  You may incorporate any other additional information to make this more impactful. Imagine that this would be included in your presentation to the customer. (Non techical audience) (5 pts)

**Do not worry if this looks like non-sense on a map.  Ex:  The trip may appear to occur over water or in a forest because this is a toy dataset.**

In [30]:
# YOUR CODE HERE:
fig_mapa = px.scatter_mapbox(df_ejemplo, lat="lat", lon="lng", color="Zone",
                             hover_name="Site", zoom=4, height=400)
fig_mapa.update_layout(mapbox_style="open-street-map")
fig_mapa.update_layout(title="Ruta Geográfica del envío")
fig_mapa.show()

# Part 3: Customer Questions


### Question 6:

The customer wants to understand the efficieny of ther DC operations.
* Based on what you see in the data, (all zones except for STORE), which parts of their operation are most & least "efficient? (10 pts)

In [36]:
# YOUR CODE HERE
tiempos_sitio = df.groupby(['asset_id', 'Site'])['time_est'].agg(['min', 'max']).reset_index()
tiempos_sitio['duracion'] = tiempos_sitio['max'] - tiempos_sitio['min']
tiempos_sitio['horas_totales'] = tiempos_sitio['duracion'].dt.total_seconds() / 3600
eficiencia_dc = tiempos_sitio[tiempos_sitio['Site'].isin(['DC 1', 'DC 2', 'Transit'])]
promedio_eficiencia = eficiencia_dc.groupby('Site')['horas_totales'].mean().reset_index()
print("Tiempo promedio de permanencia (en horas):")
print(promedio_eficiencia)


Tiempo promedio de permanencia (en horas):
      Site  horas_totales
0     DC 1       2.075482
1     DC 2      13.315480
2  Transit      41.564614


YOUR TEXT ANSWER HERE: El sitio más eficiente es DC 1 al tener menos horas totales y el menos eficiente es Transit con mas horas totales.


### Question 7:

The customer wants to understand the stocking efficiency in stores.
* Based on what you see in the data, how quickly did the store unload and stock the merchandise. (5 pts)
* How could this be converted in a KPI that a regional manager could track?  (5 pts)

In [39]:
# YOUR CODE HERE
datos_tienda = df[df['Site'] == 'Store'].copy()
kpi_tienda = datos_tienda.groupby('asset_id')['time_est'].agg(['min', 'max']).reset_index()
kpi_tienda['tiempo_proceso'] = kpi_tienda['max'] - kpi_tienda['min']
kpi_tienda['minutos_en_tienda'] = kpi_tienda['tiempo_proceso'].dt.total_seconds() / 60
promedio_kpi = kpi_tienda['minutos_en_tienda'].mean()
print(f"El tiempo promedio de procesamiento en tienda es de: {promedio_kpi:.2f} minutos")

El tiempo promedio de procesamiento en tienda es de: 1726.87 minutos


YOUR TEXT ANSWER HERE: Notemos que el tiempo promedio es de 1726.87 min, que es equivalente a 28.7 hrs, lo cual es más de un día, el gerente puede ponerse cómo meta reducir ese tiempo a 24 hrs, que seria un día.

### Question 8:

Please explain what you would ask for and what you will do with this data, given that you can talk with the following people (no code needed):


YOUR TEXT ANSWER HERE
* a. Al Gerente: Pediria que me diera laa cantidad de personas que trabajan ahí y sus horarios, así como la cantidad de carga que se recibe cada día, para así saber si el problema es falta de perosnal o falta de espacio.
* b. Al coordinador de transporte: Pediria que me de el sistema de refrigeracion de los camiones para saber si los picos de temperaturas registrados son por fallas mecanicas o factores externos.

## Part 4: Bonus Insights

### Question 8

The customer is open to hearing about additional insights you found in the data above and beyond what they asked for.
* Based on what you can see in the data, are there any other interesting insights that the customer may want to hear about? (Up to 15 bonus points)



In [53]:
# YOUR CODE HERE
estabilidad_total = df.groupby('Zone')['Temperature_C'].std().reset_index().fillna(0)
estabilidad_total = estabilidad_total.sort_values('Temperature_C', ascending=False)

fig_todas_zonas = px.bar(estabilidad_total,
                        x='Zone',
                        y='Temperature_C',
                        title='Inestabilidad Térmica por Zona',
                        labels={'Temperature_C': 'Variación de temperatura', 'Zone': 'Zona'},
                        template='plotly_white')
fig_todas_zonas.update_traces(marker_color='#e74c3c')
fig_todas_zonas.show()

YOUR TEXT ANSWER HERE
